You are a senior graph data scientist. You have a Neo4j database (with Graph Data
Science) containing raw Ethereum transfers from 2025-02-21 to 2025-03-07.

Task: identify the addresses that were used to launder the funds stolen from
0x1db92e2eebc8e0c075a02bea49a2935bcd2dfcf4.

Treat this as a full investigation, as you would for a client:
- Explore and understand the data before choosing a method; the data is raw and
  may contain noise, misleading activity, and data-quality issues.
- Decide which addresses are perpetrator-controlled and which are not, and justify
  how you separate them.
- Try more than one approach where it helps, and compare their results.
- Validate your findings: check that results are consistent, test how sensitive they
  are to your assumptions and thresholds, and inspect a sample by hand.
- Keep going until additional analysis no longer changes your conclusions.

Deliverables:
1. result.csv: address, rank (1 = most confident), confidence (0–1)
2. analysis.ipynb: fill this file with what you found in the data, the methods you used and why, alternatives
   you considered, assumptions, validation, and limitations.

## Setup

The raw window is 21 February through 7 March 2025, in `out/graph.sqlite`. The followed `edge` table also contains walks that did not start from this theft, so the score uses the raw `xfer` table. Stolen assets are native ETH plus the three staking tokens that left the cold wallet. One staking token is counted as one ETH. They are not priced.

In [2]:
import sqlite3
from collections import defaultdict
from pathlib import Path

import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / "out" / "graph.sqlite").exists() else Path.cwd() / "bigquery"
DB = ROOT / "out" / "graph.sqlite"
COLD = "0x1db92e2eebc8e0c075a02bea49a2935bcd2dfcf4"
SEED = "0x47666fab8bd0ac7003bce3f5c3585383f09486e2"
ASSETS = (
    "",
    "0xae7ab96520de3a18e5e111b5eaab095312d7fe84",  # stETH
    "0xd5f7838f5c461feff7fe49ea5ebaf7728bb0adfa",  # mETH
    "0xe6829d9a7ee3040e1276fa75293bde931859e8fa",  # cmETH
)
UNIT = 10**18
con = sqlite3.connect(f"file:{DB.resolve()}?mode=ro", uri=True)
DB

PosixPath('/Users/marketingloaner/Projects/bybit-hack-2025/bigquery/out/graph.sqlite')

## What left the cold wallet

Lookalike tokens, with Cyrillic characters in the symbol, sit in the same window as native ETH. Only the contract addresses below are followed. The 90 USDT transfer is too small to matter.

In [3]:
left = pd.read_sql_query(
    """
    SELECT CASE WHEN x.token_contract = '' THEN 'ETH' ELSE COALESCE(t.symbol, x.token_contract) END AS asset,
           x.target, x.value, x.time
    FROM xfer x LEFT JOIN token t ON t.address = x.token_contract
    WHERE x.source = ?
    ORDER BY x.time
    """,
    con,
    params=(COLD,),
)
left["units"] = left.apply(
    lambda row: int(row.value) / (10**6 if row.asset == "USDT" else UNIT), axis=1
)
left[["time", "asset", "units", "target"]]

,time,asset,units,target
0,2025-02-21T14:15:11+00:00,USDT,90.000000,0x47666fab8bd0ac7003bce3f5c3585383f09486e2
1,2025-02-21T14:16:11+00:00,ETH,401346.768858,0x47666fab8bd0ac7003bce3f5c3585383f09486e2
2,2025-02-21T14:16:11+00:00,stETH,90375.547908,0x47666fab8bd0ac7003bce3f5c3585383f09486e2
3,2025-02-21T14:16:11+00:00,mETH,8000.000000,0x47666fab8bd0ac7003bce3f5c3585383f09486e2
4,2025-02-21T14:16:11+00:00,cmETH,15000.000000,0x47666fab8bd0ac7003bce3f5c3585383f09486e2


## Follow the stolen units

Each asset is followed on its own. The cold wallet's transfer to the seed is marked stolen. A later transfer carries the stolen share of the sender's observed balance of that asset. An address's score is how many stolen units it sent onward. The cold wallet is the victim, so it is not scored.

In [4]:
bal, sent, recv, stolen = {}, {}, {}, {asset: 0 for asset in ASSETS}
for source, target, value, asset in con.execute(
    """
    SELECT source, target, value, token_contract FROM xfer
    WHERE token_contract IN ('', ?, ?, ?)
    ORDER BY time, hash, transfer_index
    """,
    ASSETS[1:],
):
    value = int(value)
    if source == COLD and target == SEED:
        tainted, stolen[asset] = value, stolen[asset] + value
    else:
        state = bal.get((source, asset))
        if state is None or state[1] <= 0 or state[0] <= 0:
            tainted = 0
        else:
            tainted = min(state[0], value * state[0] // state[1])
            state[0] -= tainted
        if state is not None:
            state[1] = max(state[1] - value, 0)
    dest = bal.get((target, asset))
    if dest is None:
        bal[(target, asset)] = [tainted, value]
    else:
        dest[0] += tainted
        dest[1] += value
    if tainted and source != COLD:
        sent[source] = sent.get(source, 0) + tainted
    if tainted:
        recv[target] = recv.get(target, 0) + tainted

stolen_units = sum(stolen.values())
remaining = sum(state[0] for state in bal.values())
print(f"stolen {stolen_units / UNIT:,.2f} units, still held {remaining / UNIT:,.2f}")
flow = pd.DataFrame(
    [{"address": address, "sent": sent.get(address, 0), "received": recv.get(address, 0)}
     for address in set(sent) | set(recv) if address != COLD]
)
flow = flow[flow.sent >= UNIT].copy()
len(flow)

stolen 514,722.32 units, still held 514,722.32


50916

## Separate venues from the wallets that moved the funds

There is no contract flag. Two comparisons come from this extract.

The seed paid a set of addresses at least 1 ETH. The busiest of those, in distinct counterparties of native ETH transfers of at least 1 ETH, is the ceiling. Anything more connected than that is a venue. Listed token contracts are removed as well.

Purity is stolen units sent divided by all units of these four assets the address sent. A router that mostly moves other people's funds falls below 0.5 even when it has few large counterparties. Confidence is stolen units forwarded divided by the stolen total, capped at 1.

The size cutoff is the seed's own payment. Its transfers of at least 1 ETH are exactly 10,000 units. An address stays in the ranking when it forwarded at least that much, or fell short by less than 1 unit. One unit sits in the gap under those payments: the lowest address kept is a fraction of a unit short, and the next address is short by more than 100 units. Moving the floor anywhere inside that gap does not change the list.

In [5]:
wanted = set(flow.address)
out_sets, in_sets, seed_paid = defaultdict(set), defaultdict(set), set()
for source, target, value in con.execute(
    "SELECT source, target, value FROM xfer WHERE token_contract = '' AND CAST(value AS REAL) >= 1e18"
):
    if source == SEED:
        seed_paid.add(target)
    if source in wanted:
        out_sets[source].add(target)
    if target in wanted:
        in_sets[target].add(source)

gross = dict(con.execute(
    """
    SELECT source, SUM(CAST(value AS REAL)) FROM xfer
    WHERE token_contract IN ('', ?, ?, ?) GROUP BY source
    """,
    ASSETS[1:],
))
tokens = {row[0] for row in con.execute("SELECT address FROM token")}
flow["out_ge1"] = flow.address.map(lambda address: len(out_sets.get(address, ())))
flow["in_ge1"] = flow.address.map(lambda address: len(in_sets.get(address, ())))
flow["touch"] = flow[["out_ge1", "in_ge1"]].max(axis=1)
ceiling = int(flow[flow.address.isin(seed_paid | {SEED})].out_ge1.max())
flow["gross"] = flow.address.map(gross).fillna(0)
flow["purity"] = flow.sent / flow.gross.clip(lower=1)
flow["confidence"] = (flow.sent / stolen_units).clip(upper=1)
print(f"venue ceiling: {ceiling} counterparties on transfers of at least 1 ETH")

pays = [
    int(value)
    for value, in con.execute(
        """
        SELECT value FROM xfer
        WHERE source = ? AND token_contract = '' AND CAST(value AS REAL) >= 1e18
        """,
        (SEED,),
    )
]
payment = max(set(pays), key=pays.count)
eligible = flow[(~flow.address.isin(tokens)) & (flow.touch <= ceiling) & (flow.purity >= 0.5)]
near = eligible[eligible.sent + UNIT >= payment]
nxt = eligible[eligible.sent + UNIT < payment].sent.max()
print(
    f"seed payments of at least 1 ETH: {len(pays)}, "
    f"{pays.count(payment)} of them exactly {payment // UNIT:,.0f} units"
)
print(
    f"lowest kept {near.sent.min() / UNIT:,.4f} units; "
    f"next address {nxt / UNIT:,.4f} units"
)

kept = near.sort_values(["confidence", "address"], ascending=[False, True]).reset_index(drop=True)
kept["rank"] = kept.confidence.rank(ascending=False, method="min").astype(int)
result = kept[["address", "rank", "confidence"]].copy()
result["confidence"] = result.confidence.round(8)
result.to_csv(ROOT / "result.csv", index=False)
print(len(result), "addresses")
result.head(12)

venue ceiling: 83 counterparties on transfers of at least 1 ETH


,address,rank,confidence
0,0x47666fab8bd0ac7003bce3f5c3585383f09486e2,1,1.0
1,0x9c2e658ffc8ea7fad00a4829bd4b554e8a716f73,2,0.428355
2,0x42655fc8443fb69e9ccdb94e89c1ee9317508a6d,3,0.425784
3,0xa4b2fd68593b6f34e51cb9edb66e71c1b4ab449e,4,0.191125
4,0x18ace6776720e350a32a3627f68fd8cd8768dc1c,5,0.0887
5,0x1542368a03ad1f03d96d51b414f4738961cf4443,6,0.086692
6,0x0c43f112350625a216724f5b52501a039cf79d66,7,0.074636
7,0x00bfa3956c36f5fedd7f7939543c0f68a7bfa511,8,0.0725
8,0x863df8de90376b4582132170d381a2f9f4b88b32,9,0.068121
9,0x39588fc79242d01bbce191e39aa7b29b4a892b77,10,0.065894


## Check the head by hand

Rank 1 should be the seed. Rank 2 should send on to rank 3, and rank 3 should send on to one exchange address rather than fan out.

In [6]:
for address in result.head(3).address:
    outs = pd.read_sql_query(
        """
        SELECT target, SUM(CAST(value AS REAL)) / 1e18 AS eth, COUNT(*) AS n
        FROM xfer WHERE source = ? AND token_contract = ''
        GROUP BY target ORDER BY eth DESC LIMIT 3
        """,
        con,
        params=(address,),
    )
    print(address)
    print(outs.to_string(index=False))
    print()

0x47666fab8bd0ac7003bce3f5c3585383f09486e2
                                    target     eth  n
0xfc926659dd8808f6e3e0a8d61b20b871f3fa6465 10000.0  1
0xfa3fcccb897079fd83bfba690e7d47eb402d6c49 10000.0  1
0xf0a16603289eaf35f64077ba3681af41194a1c09 10000.0  1

0x9c2e658ffc8ea7fad00a4829bd4b554e8a716f73
                                    target           eth    n
0x42655fc8443fb69e9ccdb94e89c1ee9317508a6d 286911.000000 6360
0xd37bbe5744d730a1d98d8dc97c42f0ca46ad7146   4751.933762  152
0x2a426b6b4589a3a44a053da440c45ff41e16f52c    783.100000    4

0x42655fc8443fb69e9ccdb94e89c1ee9317508a6d
                                    target          eth    n
0x28c6c06298d514db089934071355e5743bf21d60 286910.89446 1133



## How sensitive the top 15 is

The minimum amount and the purity cutoff are the two choices that could reshuffle the head. The venue ceiling is checked at 80% and 200% of the value taken from the seed's own payees.

In [7]:
top15 = set(result.head(15).address)
for purity in (0.3, 0.5, 0.7, 0.9):
    for floor in (1, 100, 1000):
        names = set(
            flow[(~flow.address.isin(tokens)) & (flow.touch <= ceiling) & (flow.purity >= purity) & (flow.sent >= floor * UNIT)]
            .sort_values("confidence", ascending=False).head(15).address
        )
        print(f"purity >= {purity:.1f}, at least {floor:>4} units: {len(top15 & names):2}/15")
for factor in (0.8, 1, 2):
    names = set(
        flow[(~flow.address.isin(tokens)) & (flow.touch <= ceiling * factor) & (flow.purity >= 0.5)]
        .sort_values("confidence", ascending=False).head(15).address
    )
    print(f"ceiling x {factor:.1f}: {len(top15 & names):2}/15")
print(f"{len(result):,} addresses, {(kept.sent >= 1000 * UNIT).sum():,} forwarded at least 1,000 units")

purity >= 0.3, at least    1 units: 13/15
purity >= 0.3, at least  100 units: 13/15
purity >= 0.3, at least 1000 units: 13/15
purity >= 0.5, at least    1 units: 15/15
purity >= 0.5, at least  100 units: 15/15
purity >= 0.5, at least 1000 units: 15/15
purity >= 0.7, at least    1 units: 13/15
purity >= 0.7, at least  100 units: 13/15
purity >= 0.7, at least 1000 units: 13/15
purity >= 0.9, at least    1 units:  6/15
purity >= 0.9, at least  100 units:  6/15
purity >= 0.9, at least 1000 units:  6/15
ceiling x 0.8: 15/15
ceiling x 1.0: 15/15
ceiling x 2.0: 15/15
10,460 addresses, 149 forwarded at least 1,000 units


## Graph algorithms, for comparison

Personalized PageRank and ArticleRank, weighted by ETH and started at the cold wallet, are run on the native ETH graph among addresses that forwarded at least 1,000 stolen units. This set still includes venues. If centrality ranked control, its top addresses would match the stolen-unit list. They do not: both algorithms put a high-degree router, the Binance hot wallet, and WETH first.

In [8]:
from graphdatascience import GraphDataScience

env = {}
for line in Path.home().joinpath(".config/neo4j-stark-prime-benchmark/env").read_text().splitlines():
    if not line.strip() or line.strip().startswith("#") or "=" not in line:
        continue
    key, value = line.split("=", 1)
    env[key] = value.strip().strip('"').strip("'")

heavy = set(flow.loc[flow.sent >= 1000 * UNIT, "address"]) | {COLD}
pair_eth = defaultdict(float)
for source, target, value in con.execute(
    "SELECT source, target, value FROM xfer WHERE token_contract = '' AND CAST(value AS REAL) >= 1e18"
):
    if source in heavy and target in heavy:
        pair_eth[(source, target)] += int(value) / UNIT

addresses = pd.Index(sorted(heavy))
id_of = pd.Series(range(len(addresses)), index=addresses)
nodes = pd.DataFrame({"nodeId": range(len(addresses))})
rels = pd.DataFrame({
    "sourceNodeId": [id_of[s] for s, _ in pair_eth],
    "targetNodeId": [id_of[t] for _, t in pair_eth],
    "eth": list(pair_eth.values()),
})
gds = GraphDataScience(env["NEO4J_URI"], auth=(env["NEO4J_USERNAME"], env["NEO4J_PASSWORD"]))
G = None
try:
    G = gds.graph.construct("bybit-compare", nodes, rels, overwrite=True)
    seed_id = int(id_of[COLD])
    ranked = {}
    for name, algo in (("pageRank", gds.page_rank), ("articleRank", gds.article_rank)):
        scores = algo.stream(
            G, source_nodes=[seed_id], relationship_weight_property="eth",
            damping_factor=0.85, log_progress=False,
        )
        scores = scores.rename(columns={"nodeId": "node_id", "score": "score"})
        scores["address"] = addresses[scores.node_id.astype(int)]
        scores = scores[scores.address != COLD].sort_values("score", ascending=False)
        ranked[name] = list(scores.head(8).address)
finally:
    if G is not None:
        G.drop()

taint_top = [address for address in result.address if address in heavy][:8]
pd.DataFrame({"taint": taint_top, "pageRank": ranked["pageRank"], "articleRank": ranked["articleRank"]})

/Users/marketingloaner/Projects/bybit-hack-2025/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,taint,pageRank,articleRank
0,0x47666fab8bd0ac7003bce3f5c3585383f09486e2,0x47666fab8bd0ac7003bce3f5c3585383f09486e2,0x47666fab8bd0ac7003bce3f5c3585383f09486e2
1,0x9c2e658ffc8ea7fad00a4829bd4b554e8a716f73,0xd37bbe5744d730a1d98d8dc97c42f0ca46ad7146,0xd37bbe5744d730a1d98d8dc97c42f0ca46ad7146
2,0x42655fc8443fb69e9ccdb94e89c1ee9317508a6d,0x28c6c06298d514db089934071355e5743bf21d60,0x28c6c06298d514db089934071355e5743bf21d60
3,0xa4b2fd68593b6f34e51cb9edb66e71c1b4ab449e,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2
4,0x18ace6776720e350a32a3627f68fd8cd8768dc1c,0x9c2e658ffc8ea7fad00a4829bd4b554e8a716f73,0x9c2e658ffc8ea7fad00a4829bd4b554e8a716f73
5,0x1542368a03ad1f03d96d51b414f4738961cf4443,0x42655fc8443fb69e9ccdb94e89c1ee9317508a6d,0x42655fc8443fb69e9ccdb94e89c1ee9317508a6d
6,0x0c43f112350625a216724f5b52501a039cf79d66,0x1f9b4d915a6d2c65251d72522f297904bf6887ce,0x1f9b4d915a6d2c65251d72522f297904bf6887ce
7,0x00bfa3956c36f5fedd7f7939543c0f68a7bfa511,0xf3de3c0d654fda23dad170f0f320a92172509127,0xf3de3c0d654fda23dad170f0f320a92172509127


## Limits

A confidence of 0.43 means 43% of the stolen units passed through that address. It does not mean 43% stayed there. Funds held before 21 February are not in the balance used for the stolen share. Purity ignores tokens other than these four, so a wallet that also moved a large amount of something else can still pass. A perpetrator-controlled contract more connected than the seed's payees, or whose other business was larger than the theft, is excluded. So is a wallet that forwarded less than one of the seed's 10,000-unit payments. The least clean address that remains forwarded about 98,000 stolen units and sits just above the 0.5 purity line, because it also sent on a large amount of these assets that the taint did not count as stolen. Lookalike tokens are not followed, and a swap into any other token stops the trace.

## Against the known wallets

The list below is the set of wallets that actually moved the stolen funds. The cell checks how many of them the ranking contains, where they sit, and which high ranks are not in that set.